In [ ]:
from pathlib import Path

def load_inp_as_txt(inp_path, txt_path=None):
    """
    安全地读取 Ansys .inp 文件并另存为 .txt。
    """
    inp_path = Path(inp_path)
    if txt_path is None:
        txt_path = inp_path.with_suffix(".txt")

    encodings = ["utf-8", "utf-16", "latin-1", "cp1252"]
    for enc in encodings:
        try:
            text = inp_path.read_text(encoding=enc)
            break
        except UnicodeDecodeError:
            continue
    else:
        raise RuntimeError("❌ 无法识别该 .inp 文件编码，请手动检查。")

    txt_path.write_text(text, encoding="utf-8")
    print(f"✅ 文件 {inp_path.name} 已保存为 UTF-8 格式 {txt_path.name}")

    return txt_path

# 用法
file = './FuselageActuators/AnsysFiles/Benchmark/SolutionInputDP52.inp'
load_inp_as_txt(file)

### draw first layer to 22nd layer

In [ ]:
# ---------- 1) 解析层 1..22 的连结 ----------
import re
from pathlib import Path
from collections import OrderedDict

def parse_layers_connectivity(txt_path, layers=range(1, 23), encoding='utf-8'):
    """
    遍历所有 EBLOCK 段，提取指定层(etnum)的元素连结。
    返回 OrderedDict: {layer: (elem_ids[list], node_sets[list-of-4])}
    规则：数据行前4个整型都等于某层号 -> 归入该层
    """
    txt_path = Path(txt_path)
    with txt_path.open('r', encoding=encoding, errors='ignore') as f:
        lines = f.readlines()

    want = set(int(x) for x in layers)
    out = OrderedDict((k, ([], [])) for k in sorted(want))  # layer -> (elem_ids, node_sets)

    in_eblock = False
    for raw in lines:
        s = raw.strip()
        if not s:
            continue
        if s.lower().startswith('eblock'):
            in_eblock = True
            continue
        if in_eblock and (s.startswith('/') or s.startswith('-1')):
            in_eblock = False
            continue
        if in_eblock and s.startswith('('):
            continue

        if in_eblock and re.match(r'^\d', s):
            nums = s.split()
            # 要求至少有 15 个数字（典型 19i9 导出），末尾4个是节点
            if len(nums) < 15:
                continue
            # 前4列是层号（都相等）
            try:
                first4 = list(map(int, nums[:4]))
                if not (first4[0] == first4[1] == first4[2] == first4[3]):
                    continue
                layer = first4[0]
            except ValueError:
                continue

            if layer in want:
                try:
                    # 第11个为元素号；最后4个为节点
                    elem_id = int(nums[10])
                    nodes = list(map(int, nums[-4:]))
                except ValueError:
                    continue
                out[layer][0].append(elem_id)
                out[layer][1].append(nodes)

    # 转成普通 list（避免交错引用）
    for k in list(out.keys()):
        eids, conns = out[k]
        out[k] = (list(eids), list(conns))
    return out


# ---------- 2) 绘制多层线框 ----------
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d.art3d import Line3DCollection

def _build_segments(nid2xyz, node_sets):
    segs = []
    for conn in node_sets:
        # 容忍四边形退化为三角形：去掉连续重复
        dedup = []
        for n in conn:
            if not dedup or dedup[-1] != n:
                dedup.append(int(n))
        # 映射到坐标
        pts, ok = [], True
        for nid in dedup:
            p = nid2xyz.get(nid)
            if p is None:
                ok = False
                break
            pts.append(p)
        if not ok or len(pts) < 2:
            continue
        # 闭合边
        for i in range(len(pts)):
            a, b = pts[i], pts[(i+1) % len(pts)]
            if np.allclose(a, b):
                continue
            segs.append([a, b])
    return np.asarray(segs, float) if segs else np.empty((0,2,3))

def plot_multi_layers_wireframe(nids, xyz, layers_dict,
                                cmap='tab20', linewidth=0.6, alpha=0.9,
                                figsize=(9,9), elev=18, azim=-50, save=None):
    """
    nids: (N,) 节点号；xyz: (N,3) 坐标
    layers_dict: {layer: (elem_ids, node_sets)}
    每个层用不同颜色画线框。
    """
    nids = np.asarray(nids).ravel()
    xyz  = np.asarray(xyz, float)
    nid2xyz = {int(nid): xyz[i] for i, nid in enumerate(nids)}

    # 准备颜色
    import matplotlib as mpl
    if isinstance(cmap, str):
        cmap = mpl.cm.get_cmap(cmap, max(10, len(layers_dict)))
    colors = [cmap(i % cmap.N) for i in range(len(layers_dict))]

    fig = plt.figure(figsize=figsize)
    ax = fig.add_subplot(111, projection='3d')

    handles = []
    for idx, (layer, (eids, conns)) in enumerate(layers_dict.items()):
        segs = _build_segments(nid2xyz, conns)
        if segs.size == 0:
            continue
        col = colors[idx]
        ax.add_collection3d(Line3DCollection(segs, linewidths=linewidth, alpha=alpha, colors=[col]))
        handles.append(plt.Line2D([0],[0], color=col, lw=2, label=f'Layer {layer}'))

    # 等比例坐标
    xyz_min, xyz_max = xyz.min(axis=0), xyz.max(axis=0)
    span = max((xyz_max - xyz_min).max(), 1e-9)
    ctr  = (xyz_min + xyz_max) / 2.0
    ax.set_xlim(ctr[0]-span/2, ctr[0]+span/2)
    ax.set_ylim(ctr[1]-span/2, ctr[1]+span/2)
    ax.set_zlim(ctr[2]-span/2, ctr[2]+span/2)

    ax.set_xlabel('X'); ax.set_ylabel('Y'); ax.set_zlabel('Z')
    ax.view_init(elev=elev, azim=azim)
    if handles:
        # 太多层可以把图例放外侧或限制显示数量
        ax.legend(handles=handles, loc='upper right', bbox_to_anchor=(1.25, 1.0))
    plt.tight_layout()
    if save:
        plt.savefig(save, dpi=300, bbox_inches='tight')
    plt.show()
    
txt_file = "./FuselageActuators/AnsysFiles/Benchmark/SolutionInputDP52.txt"
import re
from pathlib import Path
import numpy as np

def parse_nblock_xyz(txt_path, id_min=None, id_max=None, encoding='utf-8'):
    """
    从 Ansys 导出文本中解析 NBLOCK 段，返回:
        nids: (N,) int 节点号（按文件出现顺序，去重）
        xyz : (N,3) float 节点坐标
    规则/假设：
      - NBLOCK 后的有效数据行至少包含 4 个数：第 1 个为节点号，接下来的 3 个为 X,Y,Z。
      - 跳过形如 '(...)' 的格式行。
      - 以 '-1' 结束；遇到新块开头（NBLOCK/EBLOCK/CMBLOCK）或 '/' 也会提前结束。
      - 兼容 Fortran 'D' 指数表示（自动转换为 'E'）。
    """
    txt_path = Path(txt_path)
    lines = txt_path.read_text(encoding=encoding, errors='ignore').splitlines()

    in_block = False
    nids, xs, ys, zs = [], [], [], []

    # 一个更宽松的数值正则（整数或浮点，含科学计数法；允许 D/E 指数）
    num_pat = re.compile(r'[+-]?(?:\d+(?:\.\d*)?|\.\d+)(?:[DEde][+-]?\d+)?|[+-]?\d+')

    i = 0
    while i < len(lines):
        s = lines[i].strip()
        up = s.upper()

        # 进入 NBLOCK
        if not in_block and up.startswith('NBLOCK'):
            in_block = True
            i += 1
            continue

        if not in_block:
            i += 1
            continue

        # 在 NBLOCK 内部：
        if not s:
            i += 1
            continue
        if s.startswith('('):                # 跳过格式行 ( ... )
            i += 1
            continue
        if s.startswith('-1'):               # 典型 NBLOCK 结束
            break
        if s.startswith('/') or up.startswith(('NBLOCK', 'EBLOCK', 'CMBLOCK')):
            # 保险：遇到新块或分隔，结束 NBLOCK 解析
            break

        # 抓取该行所有“数字 token”
        tokens = [t.replace('D','E').replace('d','e') for t in num_pat.findall(s)]
        if len(tokens) < 4:
            # 有些 NBLOCK 采用“多行一条记录”（非常罕见），此处按简单行式处理；不足则跳过
            i += 1
            continue

        # 第一个应为节点号，后面三个是坐标
        try:
            nid = int(float(tokens[0]))
            x = float(tokens[1])
            y = float(tokens[2])
            z = float(tokens[3])
        except ValueError:
            i += 1
            continue

        # 过滤范围（若提供）
        if (id_min is not None and nid < int(id_min)) or (id_max is not None and nid > int(id_max)):
            i += 1
            continue

        nids.append(nid); xs.append(x); ys.append(y); zs.append(z)
        i += 1

    # 去重（保留首次出现的坐标）
    seen = set()
    uniq_nids, uniq_xyz = [], []
    for nid, x, y, z in zip(nids, xs, ys, zs):
        if nid not in seen:
            seen.add(nid)
            uniq_nids.append(int(nid))
            uniq_xyz.append((float(x), float(y), float(z)))

    nids_arr = np.asarray(uniq_nids, dtype=int)
    xyz_arr  = np.asarray(uniq_xyz, dtype=float).reshape(-1, 3)
    return nids_arr, xyz_arr
# 1) 解析 NBLOCK（你已有）得到节点号与坐标
nids, xyz = parse_nblock_xyz(txt_file, id_min=1, id_max=999999)

# 2) 解析 1..22 层的连结
layers_dict = parse_layers_connectivity(txt_file, layers=range(1, 23))

# 3) 画图（自动为每层选颜色）
plot_multi_layers_wireframe(
    nids, xyz, layers_dict,
    cmap='tab20', linewidth=0.6, alpha=0.9,
    elev=18, azim=-50, save=None  # 或 'layers_1_to_22.png'
)

In [ ]:
def plot_multi_layers_wireframe(
    nids, xyz, layers_dict,
    cmap='tab20', linewidth=0.6, alpha=0.9,
    figsize=(9,9), elev=18, azim=-50, save=None,
    # 新增：高亮参数
    highlight_eids=None, highlight_color='k',
    highlight_linewidth=1.4, highlight_alpha=1.0,
    draw_centroids=False, centroid_size=8
):
    """
    nids: (N,) 节点号；xyz: (N,3) 坐标
    layers_dict: {layer: (elem_ids, node_sets)}
    每个层用不同颜色画线框；若提供 highlight_eids，则将这些元素的边框用黑色覆盖加粗。
    """
    import numpy as np
    import matplotlib.pyplot as plt
    from mpl_toolkits.mplot3d.art3d import Line3DCollection
    import matplotlib as mpl

    nids = np.asarray(nids).ravel()
    xyz  = np.asarray(xyz, float)
    nid2xyz = {int(nid): xyz[i] for i, nid in enumerate(nids)}

    # 准备颜色
    if isinstance(cmap, str):
        cmap = mpl.cm.get_cmap(cmap, max(10, len(layers_dict)))
    colors = [cmap(i % cmap.N) for i in range(len(layers_dict))]

    # 统一把要高亮的 eid 放入 set
    hi_set = set(int(e) for e in (highlight_eids or []))

    fig = plt.figure(figsize=figsize)
    ax = fig.add_subplot(111, projection='3d')

    handles = []
    # 先画普通层
    for idx, (layer, (eids, conns)) in enumerate(layers_dict.items()):
        segs = _build_segments(nid2xyz, conns)
        if segs.size == 0:
            continue
        col = colors[idx]
        ax.add_collection3d(Line3DCollection(segs, linewidths=linewidth, alpha=alpha, colors=[col]))
        handles.append(plt.Line2D([0],[0], color=col, lw=2, label=f'Layer {layer}'))

    # 再画高亮元素（覆盖在上层）
    if hi_set:
        hi_conns = []
        hi_centroids = []
        for layer, (eids, conns) in layers_dict.items():
            for eid, conn in zip(eids, conns):
                if int(eid) in hi_set:
                    hi_conns.append(conn)
                    if draw_centroids:
                        # 计算元素几何中心（基于节点坐标平均）
                        pts = np.array([nid2xyz[n] for n in conn if n in nid2xyz], dtype=float)
                        if len(pts) > 0:
                            hi_centroids.append(pts.mean(axis=0))

        hi_segs = _build_segments(nid2xyz, hi_conns)
        if hi_segs.size > 0:
            ax.add_collection3d(Line3DCollection(
                hi_segs, linewidths=highlight_linewidth,
                alpha=highlight_alpha, colors=[highlight_color]
            ))
            handles.append(plt.Line2D([0],[0], color=highlight_color, lw=2, label='Elem 27–80'))

        if draw_centroids and len(hi_centroids) > 0:
            hi_centroids = np.array(hi_centroids, dtype=float)
            ax.scatter(hi_centroids[:,0], hi_centroids[:,1], hi_centroids[:,2],
                       s=centroid_size, c=highlight_color, depthshade=False)

    # 等比例坐标
    xyz_min, xyz_max = xyz.min(axis=0), xyz.max(axis=0)
    span = max((xyz_max - xyz_min).max(), 1e-9)
    ctr  = (xyz_min + xyz_max) / 2.0
    ax.set_xlim(ctr[0]-span/2, ctr[0]+span/2)
    ax.set_ylim(ctr[1]-span/2, ctr[1]+span/2)
    ax.set_zlim(ctr[2]-span/2, ctr[2]+span/2)

    ax.set_xlabel('X'); ax.set_ylabel('Y'); ax.set_zlabel('Z')
    ax.view_init(elev=elev, azim=azim)
    if handles:
        ax.legend(handles=handles, loc='upper right', bbox_to_anchor=(1.25, 1.0))
    plt.tight_layout()
    if save:
        plt.savefig(save, dpi=300, bbox_inches='tight')
    plt.show()

In [ ]:
layers_dict = parse_layers_connectivity(txt_file, layers=range(1, 23))

highlight_ids = list(range(27, 81))  # 27..80（含80）
plot_multi_layers_wireframe(
    nids, xyz, layers_dict,
    cmap='tab20', linewidth=0.6, alpha=0.9,
    elev=18, azim=-50, save=None,
    highlight_eids=highlight_ids,          # 新增
    highlight_color='k',                   # 黑色
    highlight_linewidth=1.4, highlight_alpha=1.0,
    draw_centroids=False                   # 如需在中心点打黑点可改为 True
)

In [ ]:
import re
import numpy as np

cmb_text = """
CMBLOCK,CM_FUSELAGE_EDGE,NODE,      177
(8i10)
         1         2         3         4         5         6         7       195
       196       197       198       199       200       201       202       203
       204       205       206       207       208       209       210       211
       212       213       214       215       216       217       218       219
       220       221       222       223       224       225       226       227
       228       229       230       231       232       233       234       235
       236       237       238       239       240       241       242       243
       244       245       246       247       248       249       250       251
       252       253       254       255       256       257       258       259
       260       261       262       263       264       265       266       267
       268       269       744       747       748       749       750       751
       752       753       754       755       756       757       760       763
       764       765       766       767       768       771       772       773
       774       775       776       779       780       781       782       783
       784       787       788       789       790       791       792       795
       796       797       798       799       800       803       804       805
       806       807       808       811       812       813       814       815
       816       819       820       821       822       823       824       827
       828       829       830       831       832       835       836       837
       838       839       840       843       844       845       846       847
       848       851       852       853       854       855       856       859
       860       861       862       863       864       865       866       867
       868
"""

# 用正则提取所有整数
nums = [int(x) for x in re.findall(r'\d+', cmb_text)]
# 去掉前两行中的 177（节点总数）
if nums and nums[0] == 177:
    nums = nums[1:]
# 转为 numpy array
edge_nodes = np.array(nums, dtype=int)

print(edge_nodes.shape)  # (177,)
print(edge_nodes[:10])   # 打印前 10 个看看

In [ ]:
def plot_multi_layers_wireframe(
    nids, xyz, layers_dict,
    cmap='tab20', linewidth=0.6, alpha=0.9,
    figsize=(9,9), elev=18, azim=-50, save=None,
    # --- 你已有的高亮元素参数 ---
    highlight_eids=None, highlight_color='k',
    highlight_linewidth=1.4, highlight_alpha=1.0,
    draw_centroids=False,
    # --- 新增：高亮节点（比如 edge_nodes） ---
    highlight_node_ids=None,
    highlight_nodes_kwargs=None  # e.g. {'s':24,'c':'k','marker':'o','depthshade':False}
):
    import numpy as np
    import matplotlib.pyplot as plt
    from mpl_toolkits.mplot3d.art3d import Line3DCollection
    import matplotlib as mpl

    nids = np.asarray(nids).ravel()
    xyz  = np.asarray(xyz, float)
    nid2xyz = {int(nid): xyz[i] for i, nid in enumerate(nids)}

    # 准备颜色
    if isinstance(cmap, str):
        cmap = mpl.cm.get_cmap(cmap, max(10, len(layers_dict)))
    colors = [cmap(i % cmap.N) for i in range(len(layers_dict))]

    fig = plt.figure(figsize=figsize)
    ax = fig.add_subplot(111, projection='3d')

    handles = []
    # 1) 画所有层
    for idx, (layer, (eids, conns)) in enumerate(layers_dict.items()):
        segs = _build_segments(nid2xyz, conns)
        if segs.size == 0:
            continue
        col = colors[idx]
        ax.add_collection3d(Line3DCollection(segs, linewidths=linewidth, alpha=alpha, colors=[col]))
        handles.append(plt.Line2D([0],[0], color=col, lw=2, label=f'Layer {layer}'))

    # 2) 画高亮元素（黑色覆盖）
    hi_set = set(int(e) for e in (highlight_eids or []))
    if hi_set:
        hi_conns = []
        for layer, (eids, conns) in layers_dict.items():
            for eid, conn in zip(eids, conns):
                if int(eid) in hi_set:
                    hi_conns.append(conn)
        hi_segs = _build_segments(nid2xyz, hi_conns)
        if hi_segs.size > 0:
            ax.add_collection3d(Line3DCollection(
                hi_segs, linewidths=highlight_linewidth, alpha=highlight_alpha, colors=[highlight_color]
            ))
            handles.append(plt.Line2D([0],[0], color=highlight_color, lw=2, label='Elem 27–80'))

    # 3) ★ 新增：标出节点（edge_nodes）
    if highlight_node_ids:
        pts = []
        for nid in highlight_node_ids:
            p = nid2xyz.get(int(nid))
            if p is not None:
                pts.append(p)
        if pts:
            pts = np.asarray(pts, float)
            sc_kwargs = dict(s=24, c='k', alpha=1.0, marker='o', depthshade=False, label='Edge nodes')
            if isinstance(highlight_nodes_kwargs, dict):
                sc_kwargs.update(highlight_nodes_kwargs)
            ax.scatter(pts[:,0], pts[:,1], pts[:,2], **sc_kwargs)
            handles.append(plt.Line2D([0],[0], color='k', marker='o', linestyle='None', label='Edge nodes'))

    # 等比例坐标与视角
    xyz_min, xyz_max = xyz.min(axis=0), xyz.max(axis=0)
    span = max((xyz_max - xyz_min).max(), 1e-9)
    ctr  = (xyz_min + xyz_max) / 2.0
    ax.set_xlim(ctr[0]-span/2, ctr[0]+span/2)
    ax.set_ylim(ctr[1]-span/2, ctr[1]+span/2)
    ax.set_zlim(ctr[2]-span/2, ctr[2]+span/2)
    ax.set_xlabel('X'); ax.set_ylabel('Y'); ax.set_zlabel('Z')
    ax.view_init(elev=elev, azim=azim)

    if handles:
        ax.legend(handles=handles, loc='upper right', bbox_to_anchor=(1.25, 1.0))
    plt.tight_layout()
    if save:
        plt.savefig(save, dpi=300, bbox_inches='tight')
    plt.show()
    return fig, ax  # 方便后续追加操作

import numpy as np

highlight_ids = list(range(27, 81))  # 27..80
edge_nodes_list = [int(x) for x in np.asarray(edge_nodes).ravel().tolist()]  # 关键：转为 list[int]

fig, ax = plot_multi_layers_wireframe(
    nids, xyz, layers_dict,
    cmap='tab20', linewidth=0.6, alpha=0.9,
    elev=18, azim=-50, save=None,
    highlight_eids=highlight_ids,
    highlight_color='w',
    highlight_linewidth=1.4, highlight_alpha=1.0,
    draw_centroids=False,
    highlight_node_ids=edge_nodes_list,  # 用 list[int]
    highlight_nodes_kwargs={'s': 24, 'c': 'k', 'alpha': 1.0, 'marker': 'o', 'depthshade': False}
)

In [ ]:
force_list = [270, 273, 279, 285, 291, 297, 303, 309, 315, 321, 327, 333, 339, 345, 351, 357, 11, 8]


In [ ]:
# -*- coding: utf-8 -*-
import re
from pathlib import Path
from collections import OrderedDict
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d.art3d import Line3DCollection
import matplotlib as mpl


# ========= 1) 解析 EBLOCK，提取 1..22 层的元素连结 =========
def parse_layers_connectivity(txt_path, layers=range(1, 23), encoding='utf-8'):
    """
    遍历所有 EBLOCK 段，提取指定层(etnum)的元素连结。
    返回 OrderedDict: {layer: (elem_ids[list], node_sets[list-of-4])}
    规则：数据行前4个整型都等于某层号 -> 归入该层
    """
    txt_path = Path(txt_path)
    with txt_path.open('r', encoding=encoding, errors='ignore') as f:
        lines = f.readlines()

    want = set(int(x) for x in layers)
    out = OrderedDict((k, ([], [])) for k in sorted(want))  # layer -> (elem_ids, node_sets)

    in_eblock = False
    for raw in lines:
        s = raw.strip()
        if not s:
            continue
        if s.lower().startswith('eblock'):
            in_eblock = True
            continue
        if in_eblock and (s.startswith('/') or s.startswith('-1')):
            in_eblock = False
            continue
        if in_eblock and s.startswith('('):
            continue

        if in_eblock and re.match(r'^\d', s):
            nums = s.split()
            # 要求至少有 15 个数字（典型 19i9 导出），末尾4个是节点
            if len(nums) < 15:
                continue
            # 前4列是层号（都相等）
            try:
                first4 = list(map(int, nums[:4]))
                if not (first4[0] == first4[1] == first4[2] == first4[3]):
                    continue
                layer = first4[0]
            except ValueError:
                continue

            if layer in want:
                try:
                    # 第11个为元素号；最后4个为节点
                    elem_id = int(nums[10])
                    nodes = list(map(int, nums[-4:]))
                except ValueError:
                    continue
                out[layer][0].append(elem_id)
                out[layer][1].append(nodes)

    # 转成普通 list
    for k in list(out.keys()):
        eids, conns = out[k]
        out[k] = (list(eids), list(conns))
    return out


# ========= 2) 解析 CMBLOCK,<name>,NODE 提取组件节点 =========
def parse_cmb_nodes(txt_path, comp_name='CM_FUSELAGE_EDGE', encoding='utf-8'):
    """
    解析 Ansys 文本中 CMBLOCK,<comp_name>,NODE 段，返回该组件的节点号 list（按出现顺序去重）。
    兼容 (8i10)/(19i9) 等格式；以 -1 或新块开始为终止。
    """
    txt_path = Path(txt_path)
    comp_name = str(comp_name).strip().upper()
    start_pat = re.compile(r'^CMBLOCK\s*,\s*([A-Za-z0-9_]+)\s*,\s*NODE', re.IGNORECASE)

    in_block = False
    out_nodes = []
    with txt_path.open('r', encoding=encoding, errors='ignore') as f:
        for raw in f:
            s = raw.strip()
            if not s:
                continue

            if not in_block:
                m = start_pat.match(s)
                if m and m.group(1).strip().upper() == comp_name:
                    in_block = True
                continue

            # 进入目标 CMBLOCK
            if s.startswith('('):
                continue
            if s.startswith('-1'):
                break
            if s.startswith('/') or s.upper().startswith(('NBLOCK', 'EBLOCK', 'CMBLOCK')):
                break

            # 行内取整数
            for token in re.findall(r'[-+]?\d+', s):
                val = int(token)
                if val == -1:
                    in_block = False
                    break
                if val > 0:
                    out_nodes.append(val)
            if not in_block:
                break

    # 去重并保持顺序
    seen, dedup = set(), []
    for n in out_nodes:
        if n not in seen:
            seen.add(n)
            dedup.append(n)
    return dedup


# ========= 3) 构造线段集合（用于线框绘制） =========
def _build_segments(nid2xyz, node_sets):
    segs = []
    for conn in node_sets:
        # 容忍四边形退化为三角形：去掉连续重复
        dedup = []
        for n in conn:
            n = int(n)
            if not dedup or dedup[-1] != n:
                dedup.append(n)
        # 映射到坐标
        pts, ok = [], True
        for nid in dedup:
            p = nid2xyz.get(nid)
            if p is None:
                ok = False
                break
            pts.append(p)
        if not ok or len(pts) < 2:
            continue
        # 闭合边
        for i in range(len(pts)):
            a, b = pts[i], pts[(i + 1) % len(pts)]
            if np.allclose(a, b):
                continue
            segs.append([a, b])
    return np.asarray(segs, float) if segs else np.empty((0, 2, 3))


# ========= 4) 绘制多层线框（支持高亮元素/边缘节点/受力节点） =========
def plot_multi_layers_wireframe(
    nids, xyz, layers_dict,
    cmap='tab20', linewidth=0.6, alpha=0.9,
    figsize=(9, 9), elev=18, azim=-50, save=None,
    # 边缘节点
    highlight_node_ids=None,
    highlight_nodes_kwargs=None,  # {'s':24,'c':'k','marker':'o','depthshade':False}
    # 受力节点
    force_node_ids=None,
    force_nodes_kwargs=None,      # {'s':60,'c':'r','marker':'*','depthshade':False}
    force_labels=False            # True: 标注节点号
):
    import numpy as np
    import matplotlib.pyplot as plt
    from mpl_toolkits.mplot3d.art3d import Line3DCollection
    import matplotlib as mpl

    nids = np.asarray(nids).ravel()
    xyz  = np.asarray(xyz, float)
    nid2xyz = {int(nid): xyz[i] for i, nid in enumerate(nids)}

    # 颜色表
    if isinstance(cmap, str):
        cmap = mpl.cm.get_cmap(cmap, max(10, len(layers_dict)))
    colors = [cmap(i % cmap.N) for i in range(len(layers_dict))]

    fig = plt.figure(figsize=figsize)
    ax = fig.add_subplot(111, projection='3d')

    legend_handles = []  # 只收集 Edge nodes / Force nodes 的图例

    # 1) 各层线框（不加入图例）
    for idx, (layer, (eids, conns)) in enumerate(layers_dict.items()):
        segs = _build_segments(nid2xyz, conns)
        if segs.size == 0:
            continue
        col = colors[idx]
        ax.add_collection3d(Line3DCollection(segs, linewidths=linewidth, alpha=alpha, colors=[col]))

    # 2) 边缘节点（黑色圆点）
    if highlight_node_ids is not None:
        try:
            highlight_node_ids = [int(x) for x in list(highlight_node_ids)]
        except Exception:
            highlight_node_ids = []
    if highlight_node_ids and len(highlight_node_ids) > 0:
        pts = []
        for nid in highlight_node_ids:
            p = nid2xyz.get(int(nid))
            if p is not None:
                pts.append(p)
        if pts:
            pts = np.asarray(pts, float)
            sc_kwargs = dict(s=24, c='k', alpha=1.0, marker='o', depthshade=False, label='Measurement Points')
            if isinstance(highlight_nodes_kwargs, dict):
                sc_kwargs.update(highlight_nodes_kwargs)
            ax.scatter(pts[:, 0], pts[:, 1], pts[:, 2], **sc_kwargs)
            # 只添加 Edge nodes 图例
            legend_handles.append(plt.Line2D([0], [0], color='k', marker='o',
                                             linestyle='None', label='Measurement Points'))

    # 3) 受力节点（红色星标）
    if force_node_ids is not None:
        try:
            force_node_ids = [int(x) for x in list(force_node_ids)]
        except Exception:
            force_node_ids = []
    if force_node_ids and len(force_node_ids) > 0:
        fpts, fids_kept = [], []
        for nid in force_node_ids:
            p = nid2xyz.get(int(nid))
            if p is not None:
                fpts.append(p)
                fids_kept.append(int(nid))
        if fpts:
            fpts = np.asarray(fpts, float)
            fk = dict(s=80, c='r', alpha=1.0, marker='*', depthshade=False, label='Applied Force Locations')
            if isinstance(force_nodes_kwargs, dict):
                fk.update(force_nodes_kwargs)
            ax.scatter(fpts[:, 0], fpts[:, 1], fpts[:, 2], **fk)
            # 只添加 Force nodes 图例
            legend_handles.append(plt.Line2D([0], [0], color='r', marker='*',
                                             linestyle='None', label='Applied Force Locations'))
            if force_labels:
                for (x_, y_, z_), nid in zip(fpts, fids_kept):
                    ax.text(x_, y_, z_, f'{nid}', fontsize=18)

    # 等比例坐标与视角
    xyz_min, xyz_max = xyz.min(axis=0), xyz.max(axis=0)
    span = max((xyz_max - xyz_min).max(), 1e-9)
    ctr  = (xyz_min + xyz_max) / 2.0
    ax.set_xlim(ctr[0] - span / 2, ctr[0] + span / 2)
    ax.set_ylim(ctr[1] - span / 2, ctr[1] + span / 2)
    ax.set_zlim(ctr[2] - span / 2, ctr[2] + span / 2)
    ax.set_xlabel('X'); ax.set_ylabel('Y'); ax.set_zlabel('Z')
    ax.view_init(elev=elev, azim=azim)

    # 只显示 Edge nodes / Force nodes 的图例
    if legend_handles:
        ax.legend(handles=legend_handles, loc='upper right', bbox_to_anchor=(1.25, 1.0))

    plt.tight_layout()
    if save:
        plt.savefig(save, dpi=300, bbox_inches='tight')
    plt.show()
    return fig, ax


# ========= 5) 示例使用 =========
if __name__ == "__main__":
    # 你的 Ansys 导出文件
    txt_file = "./FuselageActuators/AnsysFiles/Benchmark/SolutionInputDP52.txt"

    # 5.1 解析节点与坐标（NBLOCK）
    # 你已有 parse_nblock_xyz；这里示意：
    # from your_module import parse_nblock_xyz
    # nids, xyz = parse_nblock_xyz(txt_file, id_min=1, id_max=999999)
    # --- 若暂时没有，可先放两个占位 numpy 数组（请替换为真实数据） ---
    # nids = np.array([...], dtype=int)
    # xyz  = np.array([...], dtype=float).reshape(-1, 3)
    raise_if_placeholder = False  # 改 True 会抛错提醒你替换
    if raise_if_placeholder:
        raise RuntimeError("请用你已有的 parse_nblock_xyz 获取 nids, xyz，然后删除这段抛错。")

    # 5.2 解析 1..22 层连结
    layers_dict = parse_layers_connectivity(txt_file, layers=range(1, 23))

    # 5.3 从 CMBLOCK 中拿边缘组件节点（177 个）
    edge_nodes = parse_cmb_nodes(txt_file, comp_name='CM_FUSELAGE_EDGE')

    # 5.4 你要高亮的元素号（27..80）
    highlight_ids = list(range(27, 81))

    # 5.5 受力节点列表（你提供的）
    force_list = [270, 273, 279, 285, 291, 297, 303, 309, 315, 321,
                  327, 333, 339, 345, 351, 357, 11, 8]

    # 5.6 规范化 edge_nodes 为 list[int]
    edge_nodes_list = [int(x) for x in np.asarray(edge_nodes).ravel().tolist()]

    # 5.7 绘制
    # 注意：这里需要你已经准备好 nids, xyz（与 txt 对应）
    # fig, ax = plot_multi_layers_wireframe(
    #     nids, xyz, layers_dict,
    #     cmap='tab20', linewidth=0.6, alpha=0.9,
    #     elev=18, azim=-50, save=None,
    #     highlight_eids=highlight_ids,
    #     highlight_color='k',
    #     highlight_linewidth=1.4, highlight_alpha=1.0,
    #     draw_centroids=False,
    #     highlight_node_ids=edge_nodes_list,
    #     highlight_nodes_kwargs={'s': 24, 'c': 'k', 'alpha': 1.0, 'marker': 'o', 'depthshade': False},
    #     force_node_ids=force_list,
    #     force_nodes_kwargs={'s': 90, 'c': 'r', 'alpha': 1.0, 'marker': '*', 'depthshade': False},
    #     force_labels=False  # 若需要标注节点号，改 True
    # )
    pass

In [ ]:
# === 路径 ===
txt_file = "./FuselageActuators/AnsysFiles/Benchmark/SolutionInputDP52.txt"

# === 1) 从 NBLOCK 解析节点号与坐标（用你已有的方法）===
# nids, xyz = parse_nblock_xyz(txt_file, id_min=1, id_max=999999)
# 这里假设你已得到 nids, xyz

# === 2) 从 EBLOCK 解析 1..22 层连结 ===
layers_dict = parse_layers_connectivity(txt_file, layers=range(1, 23))

# === 3) 从 CMBLOCK 解析边缘组件节点（177 个）===
edge_nodes = parse_cmb_nodes(txt_file, comp_name='CM_FUSELAGE_EDGE')
edge_nodes_list = [int(x) for x in np.asarray(edge_nodes).ravel().tolist()]

# === 4) 需要高亮的元素与受力节点 ===
highlight_ids = list(range(27, 81))  # 元素号 27..80

# force_list = [270, 273, 279, 285, 291, 297, 303, 309, 315, 321,
#               327, 333, 339, 345, 351, 357, 11, 8]

force_list = [270, 8]

# === 5) 调用绘图 ===
fig, ax = plot_multi_layers_wireframe(
    nids, xyz, layers_dict,
    cmap='tab20', linewidth=0.6, alpha=0.9,
    elev=18, azim=-50, save=None,
    # # 黑色加粗覆盖元素 27–80
    # highlight_eids=highlight_ids,
    # highlight_color='k',
    # highlight_linewidth=1.4, highlight_alpha=1.0,
    # 黑色圆点标出 177 个边缘节点
    highlight_node_ids=edge_nodes_list,
    highlight_nodes_kwargs={'s': 24, 'c': 'k', 'alpha': 1.0, 'marker': 'o', 'depthshade': False},
    # 红色★标出 force_list（可切换 force_labels=True 显示编号）
    force_node_ids=force_list,
    force_nodes_kwargs={'s': 90, 'c': 'r', 'alpha': 1.0, 'marker': '*', 'depthshade': False},
    force_labels=False
)

In [ ]:
def plot_multi_layers_wireframe(
    nids, xyz, layers_dict,
    cmap='tab20', linewidth=0.6, alpha=0.9,
    figsize=(9, 9), elev=18, azim=-50, save=None,
    # 边缘节点
    highlight_node_ids=None,
    highlight_nodes_kwargs=None,  # {'s':24,'c':'k','marker':'o','depthshade':False}
    # 受力节点：分两组
    force_node_ids_main=None,     # 这组画成红色★
    force_node_ids_rest=None,     # 这组画成红色x
    force_main_kwargs=None,       # 自定义★样式
    force_rest_kwargs=None,       # 自定义x样式
    force_labels=True            # True: 给两组都标注节点号
):
    import numpy as np
    import matplotlib.pyplot as plt
    from mpl_toolkits.mplot3d.art3d import Line3DCollection
    import matplotlib as mpl

    # --- 预处理坐标映射 ---
    nids = np.asarray(nids).ravel()
    xyz  = np.asarray(xyz, float)
    nid2xyz = {int(nid): xyz[i] for i, nid in enumerate(nids)}

    # --- 颜色表，用于层线框（不进图例）---
    if isinstance(cmap, str):
        cmap = mpl.cm.get_cmap(cmap, max(10, len(layers_dict)))
    colors = [cmap(i % cmap.N) for i in range(len(layers_dict))]

    fig = plt.figure(figsize=figsize)
    ax = fig.add_subplot(111, projection='3d')

    legend_handles = []  # 我们只手动放 Edge nodes / Force nodes(main) / Force nodes(rest)

    # 1) 画所有层线框（不进图例）
    for idx, (layer, (eids, conns)) in enumerate(layers_dict.items()):
        segs = _build_segments(nid2xyz, conns)
        if segs.size == 0:
            continue
        col = colors[0]
        ax.add_collection3d(Line3DCollection(segs, linewidths=linewidth, alpha=alpha, colors=[col]))

    # 2) 边缘节点（黑色圆点）
    if highlight_node_ids is not None:
        try:
            highlight_node_ids = [int(x) for x in list(highlight_node_ids)]
        except Exception:
            highlight_node_ids = []
    if highlight_node_ids and len(highlight_node_ids) > 0:
        pts = []
        for nid in highlight_node_ids:
            p = nid2xyz.get(int(nid))
            if p is not None:
                pts.append(p)
        if pts:
            pts = np.asarray(pts, float)
            edge_kwargs = dict(s=24, c='k', alpha=1.0, marker='o', depthshade=False)
            if isinstance(highlight_nodes_kwargs, dict):
                edge_kwargs.update(highlight_nodes_kwargs)
            ax.scatter(pts[:, 0], pts[:, 1], pts[:, 2], **edge_kwargs)
            legend_handles.append(plt.Line2D([0], [0], color='k', marker='o',
                                             linestyle='None', label='Measurement Points'))

    # helper: 通用函数画一组力点
    def _plot_force_group(node_ids, style_kwargs_default, style_kwargs_user, label, add_to_legend=True):
        if node_ids is None:
            return None, []
        try:
            node_ids = [int(x) for x in list(node_ids)]
        except Exception:
            node_ids = []
        if not node_ids:
            return None, []

        pts_list, kept_ids = [], []
        for nid in node_ids:
            p = nid2xyz.get(int(nid))
            if p is not None:
                pts_list.append(p)
                kept_ids.append(int(nid))
        if not pts_list:
            return None, []

        pts_arr = np.asarray(pts_list, float)

        kw = style_kwargs_default.copy()
        if isinstance(style_kwargs_user, dict):
            kw.update(style_kwargs_user)

        ax.scatter(pts_arr[:,0], pts_arr[:,1], pts_arr[:,2], **kw)

        handle = None
        if add_to_legend:
            handle = plt.Line2D([0],[0],
                                color=kw.get('c','r'),
                                marker=kw.get('marker','*'),
                                linestyle='None',
                                label=label)
        return (pts_arr, kept_ids), ([handle] if handle else [])

    # 3) 受力节点主组（红色★，比如 [270, 8]）
    main_default = dict(s=90, c='r', alpha=1.0, marker='*', depthshade=False)
    (pts_main, ids_main), h_main = _plot_force_group(
        force_node_ids_main,
        main_default,
        force_main_kwargs,
        label='Used Force Actuators (main)',
        add_to_legend=True
    )
    legend_handles.extend(h_main)

    # 4) 受力节点其它组（红色x，比如除了270/8以外的force_list节点）
    rest_default = dict(s=60, c='r', alpha=1.0, marker='x', depthshade=False)
    (pts_rest, ids_rest), h_rest = _plot_force_group(
        force_node_ids_rest,
        rest_default,
        force_rest_kwargs,
        label='Unused Force Actuators',
        add_to_legend=True if force_node_ids_rest else False
    )
    legend_handles.extend(h_rest)

    # 5) 可选：在两组力点旁标号
    if force_labels:
        if pts_main is not None:
            for (x_, y_, z_), nid in zip(pts_main, ids_main):
                ax.text(x_, y_, z_, f'{nid}', fontsize=8, color='r')
        if pts_rest is not None:
            for (x_, y_, z_), nid in zip(pts_rest, ids_rest):
                ax.text(x_, y_, z_, f'{nid}', fontsize=8, color='r')

    # 6) 统一坐标范围、视角、图例
    xyz_min, xyz_max = xyz.min(axis=0), xyz.max(axis=0)
    span = max((xyz_max - xyz_min).max(), 1e-9)
    ctr  = (xyz_min + xyz_max) / 2.0
    ax.set_xlim(ctr[0] - span / 2, ctr[0] + span / 2)
    ax.set_ylim(ctr[1] - span / 2, ctr[1] + span / 2)
    ax.set_zlim(ctr[2] - span / 2, ctr[2] + span / 2)

    tick_fs = 14      # tick label fontsize
    label_fs = 16     # axis label fontsize
    legend_fs = 14    # legend fontsize
    ax.tick_params(axis='both', which='major', labelsize=tick_fs)
    ax.tick_params(axis='z', which='major', labelsize=tick_fs)
    
    ax.set_xlabel('X [in]', fontsize=label_fs)
    ax.set_ylabel('Y [in]', fontsize=label_fs)
    ax.set_zlabel('Z [in]', fontsize=label_fs)
    ax.view_init(elev=elev, azim=azim)
    # 图例：Edge nodes + Force nodes(main) + Force nodes(rest)
    if legend_handles:
        ax.legend(
            handles=legend_handles,
            loc='upper right',
            bbox_to_anchor=(1.25, 1.0),
            fontsize=legend_fs,      # enlarge legend text
            markerscale=1.5,         # enlarge legend markers
            frameon=True
        )

    plt.tight_layout()
    if save:
        plt.savefig(save, dpi=1400, bbox_inches='tight')
    plt.show()
    return fig, ax

In [ ]:
all_force_nodes = [270, 273, 279, 285, 291, 297, 303, 309, 315, 321,
                   327, 333, 339, 345, 351, 357, 11, 8]
# main_force = [270, 273, 279, 285, 351, 357, 11, 8]
main_force = [270, 8]
rest_force = [nid for nid in all_force_nodes if nid not in main_force]

fig, ax = plot_multi_layers_wireframe(
    nids, xyz, layers_dict,
    cmap='tab20', linewidth=0.6, alpha=0.9,
    elev=18, azim=-50, save='2fuselage.png',
    highlight_node_ids=edge_nodes_list,
    highlight_nodes_kwargs={'s': 24, 'c': 'k', 'alpha': 1.0,
                            'marker': 'o', 'depthshade': False},
    force_node_ids_main=main_force,
    force_main_kwargs={'s': 150, 'c': 'r', 'alpha': 1.0,
                       'marker': '*', 'depthshade': False},
    force_node_ids_rest=rest_force,
    force_rest_kwargs={'s': 60, 'c': 'r', 'alpha': 1.0,
                       'marker': 'x', 'depthshade': False},
    force_labels=False
)

In [1]:
for i in range(4,10):
    print(i)

4
5
6
7
8
9
